In [34]:
# # Notebook 2: Análisis Exploratorio de Datos (EDA)
# 
# Análisis comparativo de los datasets v2 y v3 de zonas AGEB.

In [35]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

OUT = 'output/'

## 1. Carga de datos

In [36]:
gdf_v2 = gpd.read_file('output/zonas_ageb_clean_v2.json')
gdf_v3 = gpd.read_file('output/zonas_ageb_clean_v3.json')

print(f"v2: {gdf_v2.shape}")
print(f"v3: {gdf_v3.shape}")

v2: (2453, 44)
v3: (2453, 42)


## 2. Estadísticas descriptivas

In [37]:
# Variables principales de riesgo
risk_cols_v2 = ['riesgo_sismo', 'riesgo_inundacion', 'riesgo_laderas', 'tipo_suelo',
                'pct_afectacion_sismo', 'pct_afectacion_inundacion', 'pct_afectacion_laderas',
                'severidad_inundacion', 'fracturas_count', 'fracturas_longitud_m',
                'ruse_emergencias', 'pob_total', 'imu_2020', 'area_total',
                'indice_riesgo_compuesto', 'riesgo_general']

risk_cols_v3 = [c for c in risk_cols_v2 if c in gdf_v3.columns]

print("=== Estadísticas v2 ===")
print(gdf_v2[risk_cols_v2].describe().T.to_string())
print("\n=== Estadísticas v3 ===")
print(gdf_v3[risk_cols_v3].describe().T.to_string())

=== Estadísticas v2 ===
                            count           mean            std          min            25%            50%            75%            max
riesgo_sismo               2453.0       4.014268       0.898726     0.000000       3.000000       4.000000       5.000000       5.000000
riesgo_inundacion          2453.0       3.560946       1.803583     0.000000       1.000000       5.000000       5.000000       5.000000
riesgo_laderas             2453.0       1.346514       0.909767     0.000000       1.000000       1.000000       1.000000       5.000000
tipo_suelo                 2453.0       2.143498       0.910669     0.000000       1.000000       2.000000       3.000000       3.000000
pct_afectacion_sismo       2453.0      88.446506       0.196837    88.072339      88.354249      88.451397      88.542189      88.824099
pct_afectacion_inundacion  2453.0      88.446506       0.196837    88.072339      88.354249      88.451397      88.542189      88.824099
pct_afectacion_la

## 3. Distribuciones de variables de riesgo

In [38]:
# Histogramas de riesgos principales
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('Distribución de Variables de Riesgo - Comparación v2 vs v3', fontsize=14, fontweight='bold')

risk_pairs = [
    ('riesgo_sismo', 'Riesgo Sísmico'),
    ('riesgo_inundacion', 'Riesgo Inundación'),
    ('riesgo_laderas', 'Riesgo Laderas'),
    ('pct_afectacion_sismo', '% Afectación Sismo'),
    ('pct_afectacion_inundacion', '% Afectación Inundación'),
    ('indice_riesgo_compuesto', 'Índice Riesgo Compuesto'),
]

for i, (col, title) in enumerate(risk_pairs):
    ax = axes[i//2, i%2]
    if col in gdf_v2.columns:
        ax.hist(gdf_v2[col], bins=30, alpha=0.5, label='v2', color='steelblue', edgecolor='white')
    if col in gdf_v3.columns:
        ax.hist(gdf_v3[col], bins=30, alpha=0.5, label='v3', color='coral', edgecolor='white')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig(f'{OUT}charts/distribuciones_riesgo_v2_v3.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: distribuciones_riesgo_v2_v3.png")

✅ Guardado: distribuciones_riesgo_v2_v3.png


In [39]:
# Distribución de riesgo_general
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, gdf, version in [(axes[0], gdf_v2, 'v2'), (axes[1], gdf_v3, 'v3')]:
    counts = gdf['riesgo_general'].value_counts().sort_index()
    colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']
    labels_map = {1:'Bajo', 2:'Bajo-Medio', 3:'Medio', 4:'Medio-Alto', 5:'Alto'}
    bars = ax.bar(counts.index, counts.values, color=colors[:len(counts)], edgecolor='white')
    ax.set_title(f'Distribución Riesgo General - {version}', fontsize=12)
    ax.set_xlabel('Nivel de Riesgo')
    ax.set_ylabel('Número de zonas AGEB')
    ax.set_xticks(counts.index)
    ax.set_xticklabels([labels_map.get(i, str(i)) for i in counts.index], rotation=45)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUT}charts/distribucion_riesgo_general.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: distribucion_riesgo_general.png")

✅ Guardado: distribucion_riesgo_general.png


In [40]:
# Boxplots de variables continuas
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Boxplots - Variables Continuas (v3)', fontsize=14, fontweight='bold')

cont_vars = ['pct_afectacion_sismo', 'pct_afectacion_inundacion', 'severidad_inundacion',
             'fracturas_count', 'ruse_emergencias', 'imu_2020']

for i, col in enumerate(cont_vars):
    ax = axes[i//3, i%3]
    if col in gdf_v3.columns:
        gdf_v3.boxplot(column=col, ax=ax)
        ax.set_title(col.replace('_', ' ').title(), fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUT}charts/boxplots_continuas_v3.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: boxplots_continuas_v3.png")

✅ Guardado: boxplots_continuas_v3.png


## 4. Análisis de correlaciones

In [41]:
# Matriz de correlación - variables de riesgo principales (v3)
corr_cols = ['riesgo_sismo', 'riesgo_inundacion', 'riesgo_laderas', 'tipo_suelo',
             'pct_afectacion_sismo', 'pct_afectacion_inundacion', 'severidad_inundacion',
             'fracturas_count', 'fracturas_longitud_m', 'ruse_emergencias', 
             'pob_total', 'imu_2020', 'area_total', 'indice_riesgo_compuesto']
corr_cols = [c for c in corr_cols if c in gdf_v3.columns]

corr_matrix = gdf_v3[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación - Variables de Riesgo (v3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}charts/correlacion_riesgos_v3.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: correlacion_riesgos_v3.png")

✅ Guardado: correlacion_riesgos_v3.png


In [42]:
# Correlaciones más fuertes con índice compuesto
print("\n=== Correlaciones con indice_riesgo_compuesto (v3) ===")
corr_idx = corr_matrix['indice_riesgo_compuesto'].drop('indice_riesgo_compuesto').sort_values(ascending=False)
print(corr_idx.to_string())


=== Correlaciones con indice_riesgo_compuesto (v3) ===
tipo_suelo                   0.934118
riesgo_inundacion            0.924504
severidad_inundacion         0.920855
riesgo_sismo                 0.913894
pct_afectacion_inundacion    0.878688
pct_afectacion_sismo         0.762369
ruse_emergencias             0.301500
fracturas_count              0.298669
imu_2020                     0.209675
fracturas_longitud_m         0.099892
pob_total                   -0.016693
area_total                  -0.178558
riesgo_laderas              -0.390274


## 5. Mapas estáticos de zonas geográficas

In [43]:
# Mapa de riesgo general v3
fig, ax = plt.subplots(figsize=(12, 12))
gdf_v3.plot(column='riesgo_general', cmap='RdYlGn_r', legend=True,
            legend_kwds={'shrink': 0.6},
            ax=ax, edgecolor='gray', linewidth=0.2, alpha=0.85)
ax.set_title('Mapa de Riesgo General por Zona AGEB (v3)', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(f'{OUT}maps/mapa_riesgo_general_v3.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: mapa_riesgo_general_v3.png")

✅ Guardado: mapa_riesgo_general_v3.png


In [44]:
# Mapas individuales por tipo de riesgo
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle('Mapas de Riesgo por Tipo - v3', fontsize=16, fontweight='bold')

for ax, col, title in zip(axes,
    ['riesgo_sismo', 'riesgo_inundacion', 'riesgo_laderas'],
    ['Riesgo Sísmico', 'Riesgo por Inundación', 'Riesgo por Laderas']):
    gdf_v3.plot(column=col, cmap='YlOrRd', legend=True,
                legend_kwds={'shrink': 0.5}, ax=ax,
                edgecolor='gray', linewidth=0.15, alpha=0.85)
    ax.set_title(title, fontsize=12)
    ax.set_axis_off()

plt.tight_layout()
plt.savefig(f'{OUT}maps/mapas_riesgo_por_tipo_v3.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: mapas_riesgo_por_tipo_v3.png")

✅ Guardado: mapas_riesgo_por_tipo_v3.png


In [45]:
# Mapa de fracturas
fig, ax = plt.subplots(figsize=(12, 12))
gdf_v3.plot(column='fracturas_count', cmap='Purples', legend=True,
            legend_kwds={'shrink': 0.6},
            ax=ax, edgecolor='gray', linewidth=0.2, alpha=0.85)
ax.set_title('Mapa de Fracturas por Zona AGEB (v3)', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(f'{OUT}maps/mapa_fracturas_v3.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: mapa_fracturas_v3.png")

✅ Guardado: mapa_fracturas_v3.png


In [46]:
# Mapa de índice compuesto
fig, ax = plt.subplots(figsize=(12, 12))
gdf_v3.plot(column='indice_riesgo_compuesto', cmap='hot_r', legend=True,
            legend_kwds={'shrink': 0.6},
            ax=ax, edgecolor='gray', linewidth=0.2, alpha=0.85)
ax.set_title('Mapa de Índice de Riesgo Compuesto (v3)', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(f'{OUT}maps/mapa_indice_compuesto_v3.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: mapa_indice_compuesto_v3.png")

✅ Guardado: mapa_indice_compuesto_v3.png


## 6. Patrones identificados

In [47]:
# Análisis por alcaldía (CVE_MUN)
print("=== Riesgo promedio por Alcaldía ===")
risk_by_mun = gdf_v3.groupby('CVE_MUN').agg({
    'riesgo_sismo': 'mean',
    'riesgo_inundacion': 'mean',
    'riesgo_laderas': 'mean',
    'fracturas_count': 'mean',
    'indice_riesgo_compuesto': 'mean',
    'riesgo_general': 'mean',
    'CVEGEO': 'count'
}).rename(columns={'CVEGEO': 'n_zonas'}).sort_values('indice_riesgo_compuesto', ascending=False)

print(risk_by_mun.round(2).to_string())

=== Riesgo promedio por Alcaldía ===
         riesgo_sismo  riesgo_inundacion  riesgo_laderas  fracturas_count  indice_riesgo_compuesto  riesgo_general  n_zonas
CVE_MUN                                                                                                                    
015              4.93               4.97            1.01            10.98                     0.75            4.76      153
017              5.00               4.96            1.01             1.87                     0.70            4.39      151
006              5.00               4.99            1.00             0.88                     0.69            4.45      110
011              4.63               4.30            1.11             1.90                     0.62            3.57      115
014              4.32               4.20            1.00             0.03                     0.62            3.62      102
005              4.20               4.15            1.36             0.20                     0

In [48]:
# Top 10 zonas más riesgosas
print("\n=== Top 10 zonas de mayor riesgo (v3) ===")
top10 = gdf_v3.nlargest(10, 'indice_riesgo_compuesto')[
    ['CVEGEO', 'CVE_MUN', 'riesgo_sismo', 'riesgo_inundacion', 'riesgo_laderas',
     'fracturas_count', 'tipo_suelo', 'indice_riesgo_compuesto', 'riesgo_general']
]
print(top10.to_string())


=== Top 10 zonas de mayor riesgo (v3) ===
            CVEGEO CVE_MUN  riesgo_sismo  riesgo_inundacion  riesgo_laderas  fracturas_count  tipo_suelo  indice_riesgo_compuesto  riesgo_general
669  0901500011002     015           5.0                5.0             1.0            47.92         3.0                 0.907312               5
500  0901500011093     015           5.0                5.0             1.0            47.92         3.0                 0.902045               5
662  0901500011110     015           5.0                5.0             1.0            47.92         3.0                 0.898625               5
539  0901500011017     015           5.0                5.0             1.0            47.92         3.0                 0.896122               5
293  0901500010818     015           5.0                5.0             1.0            47.92         3.0                 0.895865               5
215  090150001120A     015           5.0                5.0             1.0      

In [49]:
# Análisis de tipo_suelo vs riesgo
print("\n=== Riesgo compuesto promedio por tipo de suelo ===")
suelo_risk = gdf_v3.groupby('tipo_suelo_label')['indice_riesgo_compuesto'].describe()
print(suelo_risk.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
gdf_v3.boxplot(column='indice_riesgo_compuesto', by='tipo_suelo_label', ax=ax)
ax.set_title('Índice de Riesgo Compuesto por Tipo de Suelo', fontsize=12)
ax.set_xlabel('Tipo de Suelo')
ax.set_ylabel('Índice de Riesgo Compuesto')
plt.suptitle('')
plt.tight_layout()
plt.savefig(f'{OUT}charts/riesgo_por_tipo_suelo.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: riesgo_por_tipo_suelo.png")


=== Riesgo compuesto promedio por tipo de suelo ===
                     count      mean       std       min       25%       50%       75%       max
tipo_suelo_label                                                                                
Arenoso (Zona III)  1208.0  0.685215  0.055158  0.390402  0.662086  0.686916  0.707864  0.907312
Mixto (Zona II)      399.0  0.562500  0.072039  0.314791  0.518970  0.577036  0.608302  0.810477
Roca (Zona I)        846.0  0.335815  0.050148  0.000000  0.300550  0.326853  0.364940  0.537062
✅ Guardado: riesgo_por_tipo_suelo.png


In [50]:
# Scatter: pob_total vs riesgo
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(gdf_v3['pob_total'], gdf_v3['indice_riesgo_compuesto'],
                     c=gdf_v3['riesgo_general'], cmap='RdYlGn_r', 
                     alpha=0.5, s=15, edgecolor='gray', linewidth=0.3)
plt.colorbar(scatter, label='Riesgo General', shrink=0.7)
ax.set_xlabel('Población Total')
ax.set_ylabel('Índice de Riesgo Compuesto')
ax.set_title('Población vs Riesgo Compuesto', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT}charts/poblacion_vs_riesgo.png', bbox_inches='tight')
plt.close()
print("✅ Guardado: poblacion_vs_riesgo.png")

✅ Guardado: poblacion_vs_riesgo.png


## 7. Resumen de hallazgos

In [51]:
print("""
=== RESUMEN DE HALLAZGOS EDA ===

1. DISTRIBUCIÓN DE RIESGOS:
   - El riesgo sísmico es predominantemente alto (valores 3-5) en la mayoría de zonas AGEB
   - El riesgo por inundación muestra mayor variabilidad (bimodal: zonas con riesgo 0-1 y 4-5)
   - El riesgo por laderas es bajo en general (mayoría valor 1), con focos específicos

2. CORRELACIONES CLAVE:
   - riesgo_sismo y tipo_suelo muestran correlación significativa
   - Las fracturas se concentran en zonas específicas (distribución muy sesgada)
   - El índice compuesto refleja primariamente sismo e inundaciones

3. PATRONES GEOGRÁFICOS:
   - Las zonas de mayor riesgo compuesto se concentran en el oriente de la CDMX
   - Las fracturas tienen un patrón espacial muy localizado
   - El suelo tipo III (arenoso) predomina en zonas de alto riesgo

4. MEJORAS v3 vs v2:
   - pct_afectacion ahora tiene variabilidad real (antes estaba comprimido a ~88%)
   - tipo_suelo=0 fue eliminado (asignado por municipio vecino)
   - Normalización unificada en un solo esquema
""")


=== RESUMEN DE HALLAZGOS EDA ===

1. DISTRIBUCIÓN DE RIESGOS:
   - El riesgo sísmico es predominantemente alto (valores 3-5) en la mayoría de zonas AGEB
   - El riesgo por inundación muestra mayor variabilidad (bimodal: zonas con riesgo 0-1 y 4-5)
   - El riesgo por laderas es bajo en general (mayoría valor 1), con focos específicos

2. CORRELACIONES CLAVE:
   - riesgo_sismo y tipo_suelo muestran correlación significativa
   - Las fracturas se concentran en zonas específicas (distribución muy sesgada)
   - El índice compuesto refleja primariamente sismo e inundaciones

3. PATRONES GEOGRÁFICOS:
   - Las zonas de mayor riesgo compuesto se concentran en el oriente de la CDMX
   - Las fracturas tienen un patrón espacial muy localizado
   - El suelo tipo III (arenoso) predomina en zonas de alto riesgo

4. MEJORAS v3 vs v2:
   - pct_afectacion ahora tiene variabilidad real (antes estaba comprimido a ~88%)
   - tipo_suelo=0 fue eliminado (asignado por municipio vecino)
   - Normalización uni